# Test Terlebih Dahulu

In [ ]:
import requests

response = requests.post(
    "http://34.101.47.211:5678/webhook/chat-aca",
    json={
        "pesan_user":"Berapa SKS semester 1?",
        "npm_mahasiswa":"2215061109",
        "session_id":"test"
    }
)

print(response.json())

{'balasan_aca': 'Halo bestie! 👋 Berdasarkan Kurikulum Program Studi Teknik Informatika, total SKS untuk Semester 1 adalah 20 SKS yaa.', 'contexts': ['[{"response":"Total SKS untuk Semester 1 adalah 20."}]']}


# **RAGAS Dataset (25 Pertanyaan Umum)**

Menjawab pertanyaan dari 25 pertanyaan dataset

In [6]:
# ============================================================
# AUTO TEST ACA + PREPARE DATASET RAGAS
# VERSI STABIL
# ============================================================

import pandas as pd
import requests
import json
import time
import uuid

# ============================================================
# CONFIG
# ============================================================

FILE_PATH = "/content/Dataset Uji RAGAS (Test Set).xlsx"

WEBHOOK_URL = "http://34.101.47.211:5678/webhook/chat-aca"

NPM_TEST = "2215061109"

DELAY_SECONDS = 5
TIMEOUT_SECONDS = 180

OUTPUT_PATH = "/content/Dataset_Hasil_Aca_RAGAS.xlsx"

# ============================================================
# LOAD DATASET
# ============================================================

print("📂 Membaca dataset...")

df = pd.read_excel(FILE_PATH)

if "Query" not in df.columns:
    raise Exception(
        "Kolom 'Query' tidak ditemukan!"
    )

# ============================================================
# TAMBAH KOLOM
# ============================================================

for col in [
    "Answer",
    "Contexts",
    "Raw_Response",
    "Status"
]:
    if col not in df.columns:
        df[col] = ""

# ============================================================
# INFO
# ============================================================

total_questions = len(df)

print("=" * 70)
print("🤖 MEMULAI PENGUJIAN ACA")
print(f"📋 Total Pertanyaan : {total_questions}")
print("=" * 70)

# ============================================================
# LOOP PERTANYAAN
# ============================================================

for index, row in df.iterrows():

    question = row["Query"]

    if pd.isna(question):
        continue

    question = str(question).strip()

    if question == "":
        continue

    # ========================================================
    # SESSION UNIK
    # ========================================================

    session_id = f"ragas_{uuid.uuid4()}"

    payload = {
        "pesan_user": question,
        "npm_mahasiswa": NPM_TEST,
        "session_id": session_id
    }

    try:

        response = requests.post(
            WEBHOOK_URL,
            json=payload,
            timeout=TIMEOUT_SECONDS
        )

        raw_response = response.text

        # ====================================================
        # STATUS HTTP
        # ====================================================

        if response.status_code != 200:

            df.at[index, "Answer"] = (
                f"HTTP ERROR {response.status_code}"
            )

            df.at[index, "Raw_Response"] = raw_response

            df.at[index, "Status"] = (
                f"HTTP_{response.status_code}"
            )

            print(
                f"[{index+1}/{total_questions}] "
                f"❌ HTTP {response.status_code}"
            )

            continue

        # ====================================================
        # PARSE JSON
        # ====================================================

        try:
            data = response.json()

        except Exception:

            df.at[index, "Answer"] = (
                "INVALID JSON RESPONSE"
            )

            df.at[index, "Raw_Response"] = raw_response

            df.at[index, "Status"] = (
                "INVALID_JSON"
            )

            print(
                f"[{index+1}/{total_questions}] "
                f"❌ INVALID JSON"
            )

            continue

        # ====================================================
        # AMBIL JAWABAN
        # ====================================================

        answer = data.get(
            "balasan_aca",
            ""
        )

        # ====================================================
        # AMBIL CONTEXT
        # ====================================================

        contexts = data.get(
            "contexts",
            []
        )

        if not isinstance(contexts, list):
            contexts = [str(contexts)]

        # ====================================================
        # SIMPAN
        # ====================================================

        df.at[index, "Answer"] = str(answer)

        df.at[index, "Contexts"] = json.dumps(
            contexts,
            ensure_ascii=False
        )

        df.at[index, "Raw_Response"] = raw_response

        df.at[index, "Status"] = "SUCCESS"

        # ====================================================
        # LOG
        # ====================================================

        context_count = len(contexts)

        print(
            f"[{index+1}/{total_questions}] "
            f"✅ SUCCESS | Context={context_count}"
        )

        # preview context pertama
        if context_count > 0:

            preview = str(
                contexts[0]
            )[:80]

            print(
                f"   ↳ {preview}"
            )

    except requests.exceptions.Timeout:

        df.at[index, "Answer"] = "TIMEOUT"

        df.at[index, "Status"] = "TIMEOUT"

        print(
            f"[{index+1}/{total_questions}] "
            f"⏰ TIMEOUT"
        )

    except Exception as e:

        df.at[index, "Answer"] = str(e)

        df.at[index, "Status"] = "EXCEPTION"

        print(
            f"[{index+1}/{total_questions}] "
            f"❌ {e}"
        )

    # ========================================================
    # DELAY
    # ========================================================

    time.sleep(DELAY_SECONDS)

# ============================================================
# SIMPAN
# ============================================================

df.to_excel(
    OUTPUT_PATH,
    index=False
)

# ============================================================
# RINGKASAN
# ============================================================

print("\n")
print("=" * 70)
print("🎉 SELESAI")
print("=" * 70)

print("\n📊 STATUS")

print(
    df["Status"]
    .value_counts(dropna=False)
)

# ============================================================
# CEK CONTEXT
# ============================================================

print("\n📊 CEK CONTEXT")

context_kosong = (
    df["Contexts"]
    .astype(str)
    .isin(["[]", "", "nan"])
    .sum()
)

print(
    f"Context kosong : "
    f"{context_kosong}/{total_questions}"
)

print(
    f"File hasil : {OUTPUT_PATH}"
)

📂 Membaca dataset...
🤖 MEMULAI PENGUJIAN ACA
📋 Total Pertanyaan : 25
[1/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Jadwal **Pelaksanaan Perkuliahan Termasuk Ujian Akhir Semester** u
[2/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Batas akhir pembayaran UKT semester ini adalah **08 Agustus 2025**
[3/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Berdasarkan dokumen yang diberikan, **KRS (Kartu Rencana Studi)** 
[4/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Untuk mengikuti ujian susulan karena sakit, berdasarkan informasi 
[5/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Total SKS di semester 1 adalah 20."}]
[6/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Mata kuliah di semester 4 adalah sebagai berikut:\n\n**Mata Kuliah
[7/25] ✅ SUCCESS | Context=1
   ↳ [{"response":"Berdasarkan informasi yang diberikan, jumlah SKS semester antara y
[8/25] ✅ SUCCESS | Context=1
   ↳ [{"nama_mahasiswa":"Oh Sehun","npm":"2215061109","jenis_dokumen":"transkrip","se
[9/25] ✅ SUCCESS | Context=1
   ↳ [{"res

/tmp/ipykernel_121867/3145809045.py:241: UserWarning: Cell contents too long (51642), truncated to 32767 characters
  df.to_excel(
/tmp/ipykernel_121867/3145809045.py:241: UserWarning: Cell contents too long (52263), truncated to 32767 characters
  df.to_excel(


# Persiapan Environment

In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
!gcloud config set project acaris-app

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [3]:
!gcloud ai models list --region=asia-southeast1 --project=acaris-app

Using endpoint [https://asia-southeast1-aiplatform.googleapis.com/]
Listed 0 items.


In [4]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project="acaris-app", location="asia-southeast1")

model = GenerativeModel("gemini-2.5-flash")
response = model.generate_content("Halo, tes saja")
print(response.text)

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
Regional Access Boundary HTTP request failed after retries: response_data={'error': {'code': 404, 'message': 'Account not found for email: 87e00d4144|alexlsronaullmanurung@gmail.com', 'status': 'NOT_FOUND'}}, retryable_error=False


Halo! Saya siap. Ada yang bisa saya bantu?


In [5]:
!pip uninstall -y ragas langchain langchain-core langchain-community langchain-google-vertexai

!pip install -U \
ragas==0.2.15 \
langchain-google-vertexai==2.0.13 \
datasets \
google-cloud-aiplatform \
openpyxl

Found existing installation: ragas 0.2.15
Uninstalling ragas-0.2.15:
  Successfully uninstalled ragas-0.2.15
Found existing installation: langchain 0.3.30
Uninstalling langchain-0.3.30:
  Successfully uninstalled langchain-0.3.30
Found existing installation: langchain-core 0.3.86
Uninstalling langchain-core-0.3.86:
  Successfully uninstalled langchain-core-0.3.86
Found existing installation: langchain-community 0.3.31
Uninstalling langchain-community-0.3.31:
  Successfully uninstalled langchain-community-0.3.31
Found existing installation: langchain-google-vertexai 2.0.13
Uninstalling langchain-google-vertexai-2.0.13:
  Successfully uninstalled langchain-google-vertexai-2.0.13
  Using cached ragas-0.2.15-py3-none-any.whl.metadata (9.0 kB)
  Using cached langchain_google_vertexai-2.0.13-py3-none-any.whl.metadata (3.8 kB)
  Using cached langchain-1.3.11-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_core-1.4.8-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_commu

# **EVALUASI RAGAS ACA**

In [7]:
# ==========================================================
# EVALUASI RAGAS ACA
# Faithfulness
# Answer Relevancy
# Context Precision
# Context Recall
# ==========================================================

import os
import json
import pandas as pd

from datasets import Dataset

from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from langchain_google_vertexai import (
    ChatVertexAI,
    VertexAIEmbeddings
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ==========================================================
# GOOGLE CREDENTIALS
# ==========================================================

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/acaris-app-c4ae73f5cef5.json"
os.environ["GOOGLE_CLOUD_PROJECT"] = "acaris-app"

# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_excel(
    "/content/Dataset_Hasil_Aca_RAGAS.xlsx"
)

# hanya data sukses
df = df[df["Status"] == "SUCCESS"].copy()

print("Jumlah data sukses:", len(df))

# ==========================================================
# RENAME KOLOM
# ==========================================================

df.rename(
    columns={
        "Query": "question",
        "Ground Truth (Jawaban Acuan)": "reference",
        "Answer": "answer",
        "Contexts": "contexts"
    },
    inplace=True
)

# ==========================================================
# CLEAN CONTEXTS
# ==========================================================

def parse_contexts(x):

    if pd.isna(x):
        return []

    try:

        data = json.loads(x)

        if isinstance(data, list):
            return [str(i) for i in data]

        return [str(data)]

    except:
        return []

df["contexts"] = df["contexts"].apply(
    parse_contexts
)

# ==========================================================
# HAPUS DATA TANPA CONTEXT
# ==========================================================

df = df[
    df["contexts"].apply(lambda x: len(x) > 0)
].copy()

print("Data dengan context:", len(df))

# ==========================================================
# BUILD DATASET
# ==========================================================

dataset = Dataset.from_pandas(
    df[
        [
            "question",
            "answer",
            "contexts",
            "reference"
        ]
    ],
    preserve_index=False
)

# ==========================================================
# GEMINI JUDGE
# ==========================================================

judge_llm = ChatVertexAI(
    model_name="gemini-2.5-flash",
    location="asia-southeast1"
)

embedding_model = VertexAIEmbeddings(
    model_name="text-embedding-004",
    location="asia-southeast1"
)

ragas_llm = LangchainLLMWrapper(
    judge_llm
)

ragas_embeddings = (
    LangchainEmbeddingsWrapper(
        embedding_model
    )
)

# ==========================================================
# EVALUASI
# ==========================================================

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

# ==========================================================
# HASIL
# ==========================================================

result_df = result.to_pandas()

# ==========================================================
# GABUNGKAN HASIL
# ==========================================================

final_df = pd.concat(
    [
        df.reset_index(drop=True),
        result_df.reset_index(drop=True)
    ],
    axis=1
)

# ==========================================================
# SIMPAN
# ==========================================================

OUTPUT_FILE = "/content/Hasil_RAGAS_ACA.xlsx"

final_df.to_excel(
    OUTPUT_FILE,
    index=False
)

# ==========================================================
# RATA-RATA SKOR
# ==========================================================

print("\n==============================")
print("HASIL EVALUASI RAGAS")
print("==============================")

print(
    f"Faithfulness      : {final_df['faithfulness'].mean():.4f}"
)

print(
    f"Answer Relevancy  : {final_df['answer_relevancy'].mean():.4f}"
)

print(
    f"Context Precision : {final_df['context_precision'].mean():.4f}"
)

print(
    f"Context Recall    : {final_df['context_recall'].mean():.4f}"
)

print("\nFile hasil:")
print(OUTPUT_FILE)

Jumlah data sukses: 25
Data dengan context: 23


/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Evaluating:   0%|          | 0/92 [00:00<?, ?it/s]


HASIL EVALUASI RAGAS
Faithfulness      : 0.8197
Answer Relevancy  : 0.7075
Context Precision : 0.8696
Context Recall    : 0.7667

File hasil:
/content/Hasil_RAGAS_ACA.xlsx


# **Accuracy Check untuk Data Mahasiswa**

Konsep:

Chat history n8n (jawaban ACA)

               ↓

Extract angka dari jawaban (IPK, nilai, dll)

               ↓

Query PostgreSQL → ambil data asli mahasiswa

               ↓

Bandingkan → hitung accuracy %

In [8]:
!pip install psycopg2-binary pandas openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 37.3 MB/s eta 0:00:00


In [22]:
# ============================================================
# EVALUASI AKURASI ACA — DOMAIN B: DATA PERSONAL MAHASISWA
# ============================================================
#
# Strategi:
#   - Domain A (Peraturan Umum) → pakai RAGAS biasa (sudah ada)
#   - Domain B (Data Personal)  → 2 metode di file ini:
#       [1] Database-Grounded Accuracy Test  (presisi tinggi)
#       [2] LLM-as-Judge pakai RAGAS (semantic, lebih fleksibel)
#
# ERD yang dipakai:
#   users → mahasiswa → dokumen_mahasiswa
#   chatbot_sessions → chatbot_messages
# ============================================================


# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

# !pip install psycopg2-binary pandas openpyxl requests ragas langchain -q
# !pip install langchain-google-vertexai -q   # kalau pakai Vertex AI


# ============================================================
# BAGIAN 1 — KONFIGURASI
# ============================================================

import psycopg2
import pandas as pd
import requests
import json
import re
import uuid
import time
from datetime import datetime

# --- Database ---
DB_CONFIG = {
    "host":     "34.101.86.214",
    "port":     5432,
    "database": "acaris_db",
    "user":     "acaris_user",
    "password": "Mars123//",
}

# --- Chatbot webhook ---
WEBHOOK_URL = "http://34.101.47.211:5678/webhook/chat-aca"
DELAY_SECONDS  = 3
TIMEOUT_SECONDS = 180

# --- NPM mahasiswa yang akan ditest ---
# Bisa satu NPM (misal kamu sendiri) atau list NPM
NPM_LIST = [
    "2215061109",   # ganti sesuai data yang ada di DB kamu
    "2415061022", # tambah NPM lain kalau mau multi-user test
]

# --- Output ---
OUTPUT_PATH = "/content/Hasil_Evaluasi_Domain_B.xlsx"


# ============================================================
# BAGIAN 2 — AMBIL GROUND TRUTH DARI DATABASE
# ============================================================

def get_ground_truth(conn, npm: str, krs_semester=None) -> dict:
    """
    Ambil seluruh data mahasiswa dari DB sebagai ground truth.
    Sesuai ERD: users → mahasiswa → dokumen_mahasiswa
    """

    # --- Identitas + IPK + Semester ---
    identity_df = pd.read_sql("""
        SELECT
            u.name              AS nama,
            u.npm_nip           AS npm,
            m.angkatan,
            m.ipk,
            m.current_semester  AS semester_aktif
        FROM users u
        JOIN mahasiswa m ON u.id = m.user_id
        WHERE TRIM(u.npm_nip) = %s
        LIMIT 1
    """, conn, params=(npm.strip(),))

    if identity_df.empty:
        raise ValueError(f"NPM {npm} tidak ditemukan di database.")

    identity = identity_df.iloc[0].to_dict()

    # --- KHS: nilai per semester ---
    khs_df = pd.read_sql("""
        SELECT
            d.semester,
            d.isi_teks_dokumen   AS raw_khs
        FROM dokumen_mahasiswa d
        JOIN users u ON d.user_id = u.id
        WHERE LOWER(d.document_type) = 'khs'
          AND TRIM(u.npm_nip) = %s
        ORDER BY d.semester DESC
    """, conn, params=(npm.strip(),))

    # --- KRS: Filter berdasarkan semester jika ditentukan, jika tidak default terbaru ---
    if krs_semester:
        krs_query = """
            SELECT isi_teks_dokumen AS raw_krs
            FROM dokumen_mahasiswa d
            JOIN users u ON d.user_id = u.id
            WHERE LOWER(d.document_type) = 'krs'
              AND TRIM(u.npm_nip) = %s
              AND d.semester = %s
            LIMIT 1
        """
        krs_params = (npm.strip(), krs_semester)
    else:
        krs_query = """
            SELECT isi_teks_dokumen AS raw_krs
            FROM dokumen_mahasiswa d
            JOIN users u ON d.user_id = u.id
            WHERE LOWER(d.document_type) = 'krs'
              AND TRIM(u.npm_nip) = %s
            ORDER BY d.semester DESC LIMIT 1
        """
        krs_params = (npm.strip(),)

    krs_df = pd.read_sql(krs_query, conn, params=krs_params)

    # --- Transkrip ---
    transkrip_df = pd.read_sql("""
        SELECT
            d.isi_teks_dokumen   AS raw_transkrip
        FROM dokumen_mahasiswa d
        JOIN users u ON d.user_id = u.id
        WHERE LOWER(d.document_type) = 'transkrip'
          AND TRIM(u.npm_nip) = %s
        ORDER BY d.semester DESC
        LIMIT 1
    """, conn, params=(npm.strip(),))

    return {
        "npm":              identity.get("npm"),
        "nama":             identity.get("nama"),
        "angkatan":         identity.get("angkatan"),
        "ipk":              float(identity.get("ipk") or 0),
        "semester_aktif":   int(identity.get("semester_aktif") or 0),
        "khs_raw":          khs_df["raw_khs"].tolist() if not khs_df.empty else [],
        "krs_raw":          krs_df["raw_krs"].iloc[0] if not krs_df.empty else "None",
        "transkrip_raw":    transkrip_df["raw_transkrip"].iloc[0] if not transkrip_df.empty else "",
    }


# ============================================================
# BAGIAN 3 — DEFINISI TEST CASES (DOMAIN B)
# ============================================================
#
# Setiap test case punya:
#   question        → pertanyaan yang dikirim ke Aca
#   field_expected  → field dari ground_truth yang jadi acuan
#   extractor       → fungsi untuk ekstrak nilai dari jawaban Aca
#   comparator      → fungsi untuk membandingkan hasil vs ground truth
#   category        → kategori pertanyaan
# ============================================================

def extract_ipk_from_text(text: str):
    """Ekstrak angka IPK dari teks jawaban Aca."""
    patterns = [
        r'IPK\s*(?:kamu|anda|mu|kumulatif)?[\s:]*(\d+[.,]\d+)',
        r'(\d+[.,]\d+)\s*(?:itu)?(?:\s*adalah)?\s*IPK',
        r'IPK.*?(\d[.,]\d{1,2})',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return float(m.group(1).replace(',', '.'))
    # Fallback: cari angka desimal 0.xx - 4.xx
    m = re.search(r'\b([0-3]\.\d{2})\b', text)
    return float(m.group(1)) if m else None


def extract_semester_from_text(text: str):
    """Ekstrak angka semester dari teks jawaban Aca."""
    m = re.search(
        r'semester\s*(?:aktif|sekarang|ini)?[\s:ke-]*(\d+)',
        text, re.IGNORECASE
    )
    return int(m.group(1)) if m else None


def extract_nama_from_text(text: str, nama_asli: str):
    """Cek apakah nama mahasiswa ada di jawaban Aca."""
    first_name = nama_asli.split()[0].lower() if nama_asli else ""
    return first_name in text.lower()


def extract_npm_from_text(text: str, npm_asli: str):
    """Cek apakah NPM ada di jawaban."""
    return npm_asli.replace(" ", "") in text.replace(" ", "")

def extract_semester_history_from_text(text: str, total_semesters: int):
    """Memastikan jawaban mengandung penyebutan riwayat semester hingga semester saat ini."""
    # Mencari pola penyebutan semester (misal: "semester 1 sampai 7")
    # Atau sekadar memverifikasi apakah angka semester saat ini muncul
    current_semester = total_semesters
    return str(current_semester) in text and "semester" in text.lower()

def build_test_cases(ground_truth: dict) -> list:
    gt = ground_truth
    return [
        # -------------------------------------------------------
        # KATEGORI: IDENTITAS
        # -------------------------------------------------------
        {
            "category":     "Identitas",
            "question":     "Siapa nama saya?",
            "expected_val": gt["nama"],
            "extractor":    lambda text: extract_nama_from_text(text, gt["nama"]),
            "comparator":   lambda extracted, expected: extracted is True,
            "field":        "nama",
            "expected_display": gt["nama"],
        },
        {
            "category":     "Identitas",
            "question":     "Apa NPM saya?",
            "expected_val": gt["npm"],
            "extractor":    lambda text: extract_npm_from_text(text, gt["npm"]),
            "comparator":   lambda extracted, expected: extracted is True,
            "field":        "npm",
            "expected_display": gt["npm"],
        },

        # -------------------------------------------------------
        # KATEGORI: IPK & SEMESTER
        # -------------------------------------------------------
        {
            "category":     "IPK",
            "question":     "Berapa IPK saya sekarang?",
            "expected_val": gt["ipk"],
            "extractor":    extract_ipk_from_text,
            "comparator":   lambda extracted, expected: (
                extracted is not None and abs(extracted - expected) <= 0.01
            ),
            "field":        "ipk",
            "expected_display": str(gt["ipk"]),
        },
        {
            "category":     "Semester",
            "question":     "Coba ceritakan perkembangan akademik saya dari semester awal hingga saat ini?",
            "expected_val": gt["semester_aktif"],
            "extractor":    lambda text: extract_semester_history_from_text(text, gt["semester_aktif"]),
            "comparator":   lambda extracted, expected: extracted is True,
            "field":        "perkembangan_semester",
            "expected_display": f"Riwayat KHS dari sem 1 s/d {gt['semester_aktif']}",
        },

        # -------------------------------------------------------
        # KATEGORI: PROFIL LENGKAP (Diperbaiki dengan LLM Judge)
        # -------------------------------------------------------
        {
            "category":     "Profil",
            "question":     "Coba kasih lihat profil akademik saya dong!",
            "expected_val": f"Nama: {gt['nama']}, IPK: {gt['ipk']}, Sem: {gt['semester_aktif']}",
            "extractor":    lambda text: text,
            "comparator":   None,  # Menggunakan LLM Judge
            "field":        "profil_lengkap",
            "expected_display": "Profil lengkap sesuai DB",
        },

        # -------------------------------------------------------
        # KATEGORI: KRS & TRANSKRIP (Diperbaiki dengan LLM Judge)
        # -------------------------------------------------------
{
            "category":     "Seminar Usul",
            "question":     "Dengan IPK dan SKS saya sekarang, apakah saya sudah bisa mengajukan seminar usul?",
            # expected_val di sini bertindak sebagai konteks pendukung untuk LLM-as-Judge
            "expected_val": f"IPK: {gt['ipk']}, Total SKS: {gt['transkrip_raw']}, Status: Periksa syarat minimal seminar usul di buku pedoman",
            "extractor":    lambda text: text,
            "comparator":   None,  # Gunakan LLM Judge untuk menilai logika chatbot
            "field":        "syarat_seminar",
            "expected_display": "Status kelayakan seminar usul berdasarkan data personal",
        },
        {
            "category":     "Transkrip",
            "question":     "Adakah matkul yang dapat nilai C? Kalau ada, apa aku tidak bisa cumlaude?",
            "expected_val": gt["transkrip_raw"],
            "extractor":    lambda text: text,
            "comparator":   None,  # Menggunakan LLM Judge
            "field":        "transkrip_raw",
            "expected_display": "isi transkrip",
        },
    ]


# ============================================================
# BAGIAN 4 — KIRIM PERTANYAAN KE ACA
# ============================================================

def ask_aca(question: str, npm: str) -> dict:
    """Kirim pertanyaan ke webhook Aca, return dict hasil."""
    session_id = f"ragas_b_{uuid.uuid4()}"
    payload = {
        "pesan_user":    question,
        "npm_mahasiswa": npm,
        "session_id":    session_id,
    }
    try:
        resp = requests.post(
            WEBHOOK_URL, json=payload, timeout=TIMEOUT_SECONDS
        )
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "raw": resp.text}
        data = resp.json()
        return {
            "answer":   data.get("balasan_aca", ""),
            "contexts": data.get("contexts", []),
            "raw":      resp.text,
        }
    except requests.Timeout:
        return {"error": "TIMEOUT"}
    except Exception as e:
        return {"error": str(e)}


# ============================================================
# BAGIAN 5 — LLM-AS-JUDGE (untuk pertanyaan KRS/Transkrip)
# ============================================================
#
# Karena KRS/Transkrip isinya panjang dan bervariasi,
# kita minta LLM menilai apakah jawaban Aca sudah sesuai
# dengan ground truth dari DB.
# ============================================================

def llm_judge(question: str, answer: str, ground_truth_context: str) -> dict:
    """
    Gunakan Vertex AI (Gemini) untuk menilai apakah jawaban
    Aca sudah benar berdasarkan data ground truth.

    Return: {"score": 0/1, "reason": "..."}
    """
    try:
        import vertexai
        from vertexai.generative_models import GenerativeModel

        vertexai.init(project="acaris-app", location="asia-southeast1")
        model = GenerativeModel("gemini-2.5-flash")

        prompt = f"""
Kamu adalah evaluator sistem chatbot akademik.

Tugasmu: nilai apakah JAWABAN CHATBOT sudah menjawab PERTANYAAN
dengan benar berdasarkan DATA GROUND TRUTH dari database.

PERTANYAAN:
{question}

JAWABAN CHATBOT:
{answer}

DATA GROUND TRUTH (dari database, ini yang BENAR):
{ground_truth_context[:2000]}

INSTRUKSI PENILAIAN:
- Nilai 1 jika jawaban chatbot KONSISTEN dengan data ground truth
- Nilai 0 jika jawaban chatbot SALAH atau TIDAK SESUAI data ground truth
- Fokus pada akurasi data, bukan gaya bahasa

Balas HANYA dengan JSON ini (tanpa komentar lain):
{{"score": 0_atau_1, "reason": "alasan singkat dalam 1-2 kalimat"}}
"""
        response = model.generate_content(prompt)
        text = response.text.strip()
        # Bersihkan jika ada markdown
        text = re.sub(r'```json|```', '', text).strip()
        return json.loads(text)

    except Exception as e:
        return {"score": -1, "reason": f"LLM judge error: {e}"}


# ============================================================
# BAGIAN 6 — RAGAS EVALUATION (untuk Domain B)
# ============================================================
#
# Untuk pertanyaan yang bisa di-generate reference-nya dari DB,
# kita bisa pakai RAGAS answer_correctness + faithfulness.
# Reference di-generate dari data DB via LLM.
# ============================================================

def generate_reference_from_db(question: str, ground_truth: dict) -> str:
    """
    Generate 'reference answer' yang ideal dari data DB.
    Ini dipakai sebagai ground truth untuk RAGAS.
    """
    try:
        import vertexai
        from vertexai.generative_models import GenerativeModel

        vertexai.init(project="acaris-app", location="asia-southeast1")
        model = GenerativeModel("gemini-2.0-flash")

        prompt = f"""
Kamu adalah Aca, asisten akademik mahasiswa Teknik Informatika.
Berdasarkan DATA MAHASISWA berikut dari database, jawab pertanyaan ini
dengan gaya bahasa santai dan informatif.

DATA MAHASISWA (GROUND TRUTH):
- Nama       : {ground_truth['nama']}
- NPM        : {ground_truth['npm']}
- IPK        : {ground_truth['ipk']}
- Semester   : {ground_truth['semester_aktif']}
- KRS Terbaru: {ground_truth['krs_raw'][:500] if ground_truth['krs_raw'] else 'tidak ada'}

PERTANYAAN: {question}

Jawab langsung dan akurat berdasarkan data di atas.
"""
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"[Reference generation error: {e}]"


def run_ragas_evaluation(ragas_data: list) -> pd.DataFrame:
    """
    Jalankan RAGAS pada data yang sudah ada reference-nya.
    ragas_data: list of dict dengan keys:
        user_input, response, reference, retrieved_contexts
    """
    try:
        from ragas import evaluate
        from ragas.metrics import (
            answer_correctness,
            faithfulness,
            answer_relevancy,
        )
        from ragas import EvaluationDataset

        dataset = EvaluationDataset.from_list(ragas_data)

        result = evaluate(
            dataset=dataset,
            metrics=[
                answer_correctness,
                faithfulness,
                answer_relevancy,
            ],
        )
        return result.to_pandas()

    except Exception as e:
        print(f"[RAGAS Error]: {e}")
        return pd.DataFrame()


# ============================================================
# BAGIAN 7 — MAIN: JALANKAN SEMUA EVALUASI
# ============================================================

def main():
    print("=" * 65)
    print("  EVALUASI DOMAIN B — DATA PERSONAL MAHASISWA")
    print("  Chatbot Aca | Teknik Informatika Unila")
    print("=" * 65)

    conn = psycopg2.connect(**DB_CONFIG)
    print("✅ Koneksi database berhasil\n")

    all_results   = []    # hasil per test case
    ragas_data    = []    # untuk RAGAS evaluation

    for npm in NPM_LIST:
        print(f"\n{'─'*50}")
        print(f"📋 NPM: {npm}")
        print(f"{'─'*50}")

        # 1. Ambil ground truth dari DB
        try:
            gt = get_ground_truth(conn, npm)
            print(f"   Nama     : {gt['nama']}")
            print(f"   IPK      : {gt['ipk']}")
            print(f"   Semester : {gt['semester_aktif']}")
        except ValueError as e:
            print(f"   ⚠️  {e} — skip")
            continue

        # 2. Build test cases
        test_cases = build_test_cases(gt)
        print(f"   📝 {len(test_cases)} test cases akan dijalankan\n")

        for i, tc in enumerate(test_cases, 1):
            q        = tc["question"]
            category = tc["category"]
            field    = tc["field"]
            expected = tc["expected_val"]

            print(f"   [{i}/{len(test_cases)}] [{category}] {q}")

            # 3. Tanya Aca
            time.sleep(DELAY_SECONDS)
            res = ask_aca(q, npm)

            if "error" in res:
                print(f"         ❌ ERROR: {res['error']}")
                all_results.append({
                    "npm":       npm,
                    "nama":      gt["nama"],
                    "category":  category,
                    "question":  q,
                    "field":     field,
                    "expected":  tc["expected_display"],
                    "extracted": None,
                    "correct":   False,
                    "method":    "regex",
                    "llm_score": None,
                    "llm_reason":res.get("error"),
                    "answer_preview": "",
                })
                continue

            answer   = res["answer"]
            contexts = res["contexts"]

            # 4. Evaluasi
            if tc["comparator"] is not None:
                # Metode regex/rule-based
                extracted = tc["extractor"](answer)
                is_correct = tc["comparator"](extracted, expected)
                llm_score  = None
                llm_reason = None
                method     = "regex"
            else:
                # Metode LLM-as-Judge
                extracted  = answer
                judge_res  = llm_judge(q, answer, str(expected)[:1500])
                is_correct = judge_res.get("score", 0) == 1
                llm_score  = judge_res.get("score")
                llm_reason = judge_res.get("reason")
                method     = "llm_judge"

            status_icon = "✅" if is_correct else "❌"
            print(f"         {status_icon} {'BENAR' if is_correct else 'SALAH'} | method={method}")
            if llm_reason:
                print(f"         💬 {llm_reason}")

            all_results.append({
                "npm":          npm,
                "nama":         gt["nama"],
                "category":     category,
                "question":     q,
                "field":        field,
                "expected":     tc["expected_display"],
                "extracted":    str(extracted)[:200],
                "correct":      is_correct,
                "method":       method,
                "llm_score":    llm_score,
                "llm_reason":   llm_reason,
                "answer_preview": str(answer)[:300],
            })

            # 5. Siapkan data RAGAS (untuk pertanyaan yang cocok)
            if tc["comparator"] is not None:
                # Generate reference dari DB
                reference = generate_reference_from_db(q, gt)
                ragas_data.append({
                    "user_input":        q,
                    "response":          answer,
                    "reference":         reference,
                    "retrieved_contexts": contexts if contexts else [""],
                })

    conn.close()

    # ============================================================
    # HASIL ACCURACY PER KATEGORI
    # ============================================================

    result_df = pd.DataFrame(all_results)

    print("\n\n" + "=" * 65)
    print("  RINGKASAN HASIL EVALUASI DOMAIN B")
    print("=" * 65)

    if not result_df.empty:
        # Per kategori
        summary = result_df.groupby("category")["correct"].agg(
            total="count",
            benar="sum"
        )
        summary["accuracy_%"] = (summary["benar"] / summary["total"] * 100).round(2)

        print("\n📊 Akurasi Per Kategori:")
        print(summary.to_string())

        # Overall
        total   = len(result_df)
        benar   = result_df["correct"].sum()
        overall = benar / total * 100 if total > 0 else 0
        print(f"\n📈 Overall Accuracy: {benar}/{total} = {overall:.2f}%")

    # ============================================================
    # RAGAS EVALUATION
    # ============================================================

    if ragas_data:
        print(f"\n\n{'─'*50}")
        print(f"🔍 Menjalankan RAGAS pada {len(ragas_data)} pertanyaan...")
        ragas_df = run_ragas_evaluation(ragas_data)

        if not ragas_df.empty:
            print("\n📊 RAGAS Metrics (Domain B):")
            for col in ["answer_correctness", "faithfulness", "answer_relevancy"]:
                if col in ragas_df.columns:
                    print(f"   {col:25s}: {ragas_df[col].mean():.4f}")

    # ============================================================
    # SIMPAN KE EXCEL
    # ============================================================

    with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
        if not result_df.empty:
            result_df.to_excel(
                writer, sheet_name="Detail_Hasil", index=False
            )
            if not result_df.empty:
                summary.reset_index().to_excel(
                    writer, sheet_name="Summary_Per_Kategori", index=False
                )
        if ragas_data and not ragas_df.empty:
            ragas_df.to_excel(
                writer, sheet_name="RAGAS_Scores", index=False
            )

    print(f"\n✅ Hasil disimpan: {OUTPUT_PATH}")
    print("=" * 65)


if __name__ == "__main__":
    main()


# ============================================================
# BONUS: MULTI-NPM TEST (ambil semua NPM dari DB)
# ============================================================
# Kalau mau test semua mahasiswa yang ada di DB:
#
# conn = psycopg2.connect(**DB_CONFIG)
# npm_df = pd.read_sql(
#     "SELECT TRIM(npm_nip) AS npm FROM users WHERE role = 'mahasiswa'",
#     conn
# )
# NPM_LIST = npm_df["npm"].tolist()
# conn.close()
#
# Lalu jalankan main() seperti biasa.
# ============================================================

  EVALUASI DOMAIN B — DATA PERSONAL MAHASISWA
  Chatbot Aca | Teknik Informatika Unila
✅ Koneksi database berhasil


──────────────────────────────────────────────────
📋 NPM: 2215061109
──────────────────────────────────────────────────


/tmp/ipykernel_121867/3690167793.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  identity_df = pd.read_sql("""
/tmp/ipykernel_121867/3690167793.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  khs_df = pd.read_sql("""
/tmp/ipykernel_121867/3690167793.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  krs_df = pd.read_sql(krs_query, conn, params=krs_params)
/tmp/ipykernel_121867/3690167793.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2

   Nama     : Alexis Ronauli Manurung
   IPK      : 3.87
   Semester : 8
   📝 7 test cases akan dijalankan

   [1/7] [Identitas] Siapa nama saya?
         ✅ BENAR | method=regex


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


   [2/7] [Identitas] Apa NPM saya?
         ✅ BENAR | method=regex
   [3/7] [IPK] Berapa IPK saya sekarang?
         ✅ BENAR | method=regex
   [4/7] [Semester] Coba ceritakan perkembangan akademik saya dari semester awal hingga saat ini?
         ✅ BENAR | method=regex
   [5/7] [Profil] Coba kasih lihat profil akademik saya dong!
         ❌ SALAH | method=llm_judge
         💬 Informasi semester yang diberikan chatbot (Semester 7) tidak sesuai dengan data ground truth (Sem: 8).
   [6/7] [Seminar Usul] Dengan IPK dan SKS saya sekarang, apakah saya sudah bisa mengajukan seminar usul?
         ✅ BENAR | method=llm_judge
         💬 Chatbot berhasil mengambil dan menyajikan data IPK (3.87) dan SKS (143) dari ground truth dengan akurat, dan menggunakannya untuk menjawab pertanyaan secara relevan.
   [7/7] [Transkrip] Adakah matkul yang dapat nilai C? Kalau ada, apa aku tidak bisa cumlaude?
         ✅ BENAR | method=llm_judge
         💬 Chatbot benar menyatakan tidak ada nilai 'C' dan nilai te

/tmp/ipykernel_121867/3690167793.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  identity_df = pd.read_sql("""
/tmp/ipykernel_121867/3690167793.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  khs_df = pd.read_sql("""
/tmp/ipykernel_121867/3690167793.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  krs_df = pd.read_sql(krs_query, conn, params=krs_params)
/tmp/ipykernel_121867/3690167793.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2

   Nama     : Fathan Rizqi Syahbana
   IPK      : 3.82
   Semester : 4
   📝 7 test cases akan dijalankan

   [1/7] [Identitas] Siapa nama saya?
         ✅ BENAR | method=regex


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


   [2/7] [Identitas] Apa NPM saya?
         ✅ BENAR | method=regex
   [3/7] [IPK] Berapa IPK saya sekarang?
         ✅ BENAR | method=regex
   [4/7] [Semester] Coba ceritakan perkembangan akademik saya dari semester awal hingga saat ini?
         ✅ BENAR | method=regex
   [5/7] [Profil] Coba kasih lihat profil akademik saya dong!
         ✅ BENAR | method=llm_judge
         💬 Semua data yang disediakan dalam ground truth (Nama, IPK, dan Semester saat ini) konsisten dengan jawaban chatbot.
   [6/7] [Seminar Usul] Dengan IPK dan SKS saya sekarang, apakah saya sudah bisa mengajukan seminar usul?
         ✅ BENAR | method=llm_judge
         💬 Chatbot berhasil mengambil data IPK dan SKS mahasiswa dengan benar dari DATA GROUND TRUTH. Chatbot juga secara jujur menyatakan ketidakmampuannya menemukan informasi mengenai syarat minimal IPK dan SKS untuk pengajuan seminar usul, yang mana informasi tersebut tidak tersedia dalam DATA GROUND TRUTH yang diberikan.
   [7/7] [Transkrip] Adakah matkul ya